# DKD Risk Prediction in Type 2 Diabetes
## 04. Performance Evaluation and Statistical Comparisons

This notebook evaluates the frozen All of Us models produced by Notebooks 03B and 03C.

No model fitting, feature selection, hyperparameter optimization, or threshold tuning
is performed here.

Primary analyses use the prespecified 90% specificity policy. Additional analyses assess
robustness at 80%, 85%, and 95% specificity, quantitative calibration, a parsimonious
clinical benchmark, and algorithmic sensitivity at the one-year landmark.


# 1. Setup


In [ ]:
import json
import math
import warnings
from pathlib import Path

import gcsfs
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from scipy.special import expit, logit
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)

warnings.filterwarnings("ignore")

migration_folder = "rw-migration-aou-rw-639fade3"
bucket = migration_folder

RESULT_DIR = f"{bucket}/aou_dkd/dkd_model_results_20260822"

fs = gcsfs.GCSFileSystem()

ID_COL = "person_id"
Y_COL = "y_true"

FINAL_DATE = "20260822"
TARGET_SPECIFICITY = 0.90
SPEC_LEVELS = [0.80, 0.85, 0.90, 0.95]
PRIMARY_PROB_COL = "p_calibrated"

N_BOOT = 2000
BOOT_SEED = 42

MODEL_ORDER = {
    "30d": ["M1", "M2", "M3", "M4a", "M5"],
    "6m":  ["M1", "M2", "M3", "M4a", "M5"],
    "1y":  ["M1", "M2", "M3", "M4",  "M5"],
    "2y":  ["M1", "M2", "M3", "M4",  "M5"],
}

LANDMARK_ORDER = ["30d", "6m", "1y", "2y"]

print("Primary probability:", PRIMARY_PROB_COL)
print("Fixed-policy target specificity:", TARGET_SPECIFICITY)
print("Bootstrap replicates:", N_BOOT)


## I/O helpers


In [ ]:
def read_gcs_csv(path):
    with fs.open(path, "r") as f:
        return pd.read_csv(f)

def save_gcs_csv(df, path):
    with fs.open(path, "w") as f:
        df.to_csv(f, index=False)
    print(f"Saved CSV: {path}")

def save_gcs_json(obj, path):
    with fs.open(path, "w") as f:
        json.dump(obj, f, indent=2)
    print(f"Saved JSON: {path}")


# 2. Load Frozen Predictions from Notebook 03B


In [ ]:
PREDICTION_PATHS = {
    landmark: {
        model_id: (
            f"{RESULT_DIR}/"
            f"dkd_{landmark}_{model_id}_predictions_{FINAL_DATE}.csv"
        )
        for model_id in MODEL_ORDER[landmark]
    }
    for landmark in LANDMARK_ORDER
}

predictions = {}

for landmark in LANDMARK_ORDER:
    predictions[landmark] = {}

    for model_id, path in PREDICTION_PATHS[landmark].items():
        if not fs.exists(path):
            raise FileNotFoundError(path)

        df = read_gcs_csv(path)

        required = {
            ID_COL,
            "split",
            Y_COL,
            "p_raw",
            "p_calibrated",
        }

        missing = required - set(df.columns)
        if missing:
            raise ValueError(
                f"{landmark}/{model_id}: missing columns {sorted(missing)}"
            )

        df[ID_COL] = pd.to_numeric(
            df[ID_COL],
            errors="raise",
        ).astype("int64")

        df[Y_COL] = pd.to_numeric(
            df[Y_COL],
            errors="raise",
        ).astype(int)

        if df[ID_COL].duplicated().any():
            raise ValueError(
                f"{landmark}/{model_id}: duplicate person_id."
            )

        if not set(df["split"]).issubset({"cal", "test"}):
            raise ValueError(
                f"{landmark}/{model_id}: unexpected split values."
            )

        for col in ["p_raw", "p_calibrated"]:
            if df[col].isna().any():
                raise ValueError(
                    f"{landmark}/{model_id}: missing {col}."
                )

            if ((df[col] < 0) | (df[col] > 1)).any():
                raise ValueError(
                    f"{landmark}/{model_id}: {col} outside [0,1]."
                )

        predictions[landmark][model_id] = df

        print(
            f"{landmark:>3} | {model_id:>3} | "
            f"CAL={int((df['split']=='cal').sum()):,} | "
            f"TEST={int((df['split']=='test').sum()):,}"
        )


# 2B. Load One-Year Clinical and Algorithmic Comparator Predictions


In [ ]:
COMPARATOR_PATHS = {
    "Clinical LR": (
        f"{RESULT_DIR}/"
        f"dkd_1y_clinical_lr_predictions_{FINAL_DATE}.csv"
    ),
    "M5 LR": (
        f"{RESULT_DIR}/"
        f"dkd_1y_M5_lr_predictions_{FINAL_DATE}.csv"
    ),
    "M5 RF": (
        f"{RESULT_DIR}/"
        f"dkd_1y_M5_rf_predictions_{FINAL_DATE}.csv"
    ),
}

comparator_predictions = {}

for label, path in COMPARATOR_PATHS.items():
    if not fs.exists(path):
        raise FileNotFoundError(
            f"{path}\nRun Notebook 03C before Notebook 04."
        )

    df = read_gcs_csv(path)

    required = {
        ID_COL,
        "split",
        Y_COL,
        "p_raw",
        "p_calibrated",
    }

    missing = required - set(df.columns)

    if missing:
        raise ValueError(
            f"{label}: missing columns {sorted(missing)}"
        )

    df[ID_COL] = pd.to_numeric(
        df[ID_COL],
        errors="raise",
    ).astype("int64")

    df[Y_COL] = pd.to_numeric(
        df[Y_COL],
        errors="raise",
    ).astype(int)

    comparator_predictions[label] = df

    print(
        f"{label:<12} | "
        f"CAL={int((df['split']=='cal').sum()):,} | "
        f"TEST={int((df['split']=='test').sum()):,}"
    )

# All one-year comparators must use the exact M5 CAL/TEST participants.
for split_name in ["cal", "test"]:
    ref = (
        predictions["1y"]["M5"]
        .loc[
            predictions["1y"]["M5"]["split"].eq(split_name),
            [ID_COL, Y_COL],
        ]
        .sort_values(ID_COL)
        .reset_index(drop=True)
    )

    for label, df in comparator_predictions.items():
        cur = (
            df.loc[
                df["split"].eq(split_name),
                [ID_COL, Y_COL],
            ]
            .sort_values(ID_COL)
            .reset_index(drop=True)
        )

        if not ref.equals(cur):
            raise ValueError(
                f"{label}: {split_name} participants/outcomes "
                "do not match one-year M5."
            )

print("One-year comparator participant/outcome identity: PASSED")


# 3. Lock Participant and Outcome Identity Across Models


In [ ]:
def sorted_split_frame(df, split_name):
    return (
        df.loc[
            df["split"].eq(split_name),
            [ID_COL, Y_COL],
        ]
        .sort_values(ID_COL)
        .reset_index(drop=True)
    )

for landmark in LANDMARK_ORDER:
    reference_model = MODEL_ORDER[landmark][0]

    for split_name in ["cal", "test"]:
        ref = sorted_split_frame(
            predictions[landmark][reference_model],
            split_name,
        )

        for model_id in MODEL_ORDER[landmark][1:]:
            cur = sorted_split_frame(
                predictions[landmark][model_id],
                split_name,
            )

            if not ref.equals(cur):
                raise ValueError(
                    f"{landmark}/{split_name}: participant/outcome "
                    f"identity differs between {reference_model} and {model_id}."
                )

print("Within-landmark CAL/TEST participant and outcome identity: PASSED")

# 30d and 6m were harmonized upstream.
for split_name in ["cal", "test"]:
    x30 = sorted_split_frame(
        predictions["30d"]["M1"],
        split_name,
    )
    x6 = sorted_split_frame(
        predictions["6m"]["M1"],
        split_name,
    )

    if not x30.equals(x6):
        raise ValueError(
            f"30d and 6m {split_name} participants/outcomes are not identical."
        )

print("30d/6m CAL and TEST identity: PASSED")


# 4. Metric Helpers


In [ ]:
def safe_auroc(y, p):
    y = np.asarray(y)
    p = np.asarray(p)
    if np.unique(y).size < 2:
        return np.nan
    return roc_auc_score(y, p)

def safe_auprc(y, p):
    y = np.asarray(y)
    p = np.asarray(p)
    if np.unique(y).size < 2:
        return np.nan
    return average_precision_score(y, p)

def confusion_metrics(y, p, threshold):
    y = np.asarray(y).astype(int)
    p = np.asarray(p, dtype=float)

    pred = (p >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y,
        pred,
        labels=[0, 1],
    ).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) else np.nan
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    ppv = tp / (tp + fp) if (tp + fp) else np.nan
    npv = tn / (tn + fn) if (tn + fn) else np.nan
    accuracy = (tp + tn) / len(y) if len(y) else np.nan

    return {
        "TP": int(tp),
        "FP": int(fp),
        "TN": int(tn),
        "FN": int(fn),
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "PPV": ppv,
        "NPV": npv,
        "Accuracy": accuracy,
    }

def choose_threshold_for_specificity(
    y,
    p,
    target_specificity=0.90,
):
    """
    Choose the lowest threshold that attains specificity >= target
    on CAL. Among eligible thresholds, this maximizes sensitivity.
    """
    y = np.asarray(y).astype(int)
    p = np.asarray(p, dtype=float)

    fpr, tpr, thresholds = roc_curve(y, p)
    specificity = 1.0 - fpr

    eligible = np.where(
        specificity >= target_specificity
    )[0]

    if len(eligible) == 0:
        raise RuntimeError(
            "No calibration threshold achieves target specificity."
        )

    # Highest sensitivity; if tied, lowest threshold.
    max_tpr = np.max(tpr[eligible])
    candidates = eligible[
        np.isclose(
            tpr[eligible],
            max_tpr,
            rtol=0,
            atol=1e-12,
        )
    ]

    chosen_idx = candidates[
        np.argmin(
            thresholds[candidates]
        )
    ]

    return {
        "threshold": float(thresholds[chosen_idx]),
        "cal_sensitivity": float(tpr[chosen_idx]),
        "cal_specificity": float(specificity[chosen_idx]),
    }

def calibration_intercept_slope(y, p):
    """
    Logistic calibration model:
        logit(P(Y=1)) = intercept + slope * logit(predicted probability)
    """
    y = np.asarray(y).astype(float)
    p = np.asarray(p, dtype=float)

    eps = 1e-6
    p = np.clip(p, eps, 1 - eps)
    z = logit(p)

    # Newton-Raphson logistic regression with intercept + fixed predictor z.
    X = np.column_stack(
        [np.ones(len(z)), z]
    )

    beta = np.array([0.0, 1.0], dtype=float)

    for _ in range(100):
        eta = X @ beta
        mu = expit(eta)
        w = np.clip(mu * (1 - mu), 1e-9, None)

        score = X.T @ (y - mu)
        hessian = -(X.T * w) @ X

        step = np.linalg.solve(
            hessian,
            score,
        )

        beta_new = beta - step

        if np.max(np.abs(beta_new - beta)) < 1e-10:
            beta = beta_new
            break

        beta = beta_new

    return float(beta[0]), float(beta[1])

def oe_ratio(y, p):
    observed = np.sum(y)
    expected = np.sum(p)
    if expected <= 0:
        return np.nan
    return observed / expected


def calibration_in_the_large(y, p):
    """
    Calibration-in-the-large (CITL): intercept-only recalibration
    with logit(predicted probability) treated as an offset.
    Ideal value = 0.
    """
    from scipy.optimize import brentq

    y = np.asarray(y).astype(float)
    p = np.asarray(p, dtype=float)

    eps = 1e-6
    p = np.clip(p, eps, 1 - eps)
    z = logit(p)

    target = float(np.sum(y))

    def score(alpha):
        return float(
            np.sum(expit(z + alpha))
            - target
        )

    return float(
        brentq(
            score,
            -30.0,
            30.0,
        )
    )


# 5. Bootstrap Confidence Intervals


In [ ]:
def stratified_bootstrap_indices(y, rng):
    y = np.asarray(y).astype(int)

    idx0 = np.where(y == 0)[0]
    idx1 = np.where(y == 1)[0]

    b0 = rng.choice(
        idx0,
        size=len(idx0),
        replace=True,
    )

    b1 = rng.choice(
        idx1,
        size=len(idx1),
        replace=True,
    )

    out = np.concatenate([b0, b1])
    rng.shuffle(out)
    return out

def percentile_ci(values, alpha=0.05):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return np.nan, np.nan

    return (
        float(np.quantile(values, alpha / 2)),
        float(np.quantile(values, 1 - alpha / 2)),
    )

def bootstrap_test_metrics(
    y,
    p,
    threshold,
    *,
    n_boot=N_BOOT,
    seed=BOOT_SEED,
):
    y = np.asarray(y).astype(int)
    p = np.asarray(p, dtype=float)

    rng = np.random.default_rng(seed)

    samples = {
        "AUROC": [],
        "AUPRC": [],
        "Brier": [],
        "O_E": [],
        "Calibration_intercept": [],
        "Calibration_slope": [],
        "CITL": [],
        "Sensitivity": [],
        "Specificity": [],
        "PPV": [],
        "NPV": [],
    }

    for _ in range(n_boot):
        idx = stratified_bootstrap_indices(
            y,
            rng,
        )

        yb = y[idx]
        pb = p[idx]

        samples["AUROC"].append(
            safe_auroc(yb, pb)
        )

        samples["AUPRC"].append(
            safe_auprc(yb, pb)
        )

        samples["Brier"].append(
            brier_score_loss(yb, pb)
        )

        samples["O_E"].append(
            oe_ratio(yb, pb)
        )

        try:
            intercept, slope = calibration_intercept_slope(
                yb,
                pb,
            )
        except Exception:
            intercept, slope = np.nan, np.nan

        samples["Calibration_intercept"].append(
            intercept
        )
        samples["Calibration_slope"].append(
            slope
        )

        try:
            citl = calibration_in_the_large(
                yb,
                pb,
            )
        except Exception:
            citl = np.nan

        samples["CITL"].append(
            citl
        )

        cm = confusion_metrics(
            yb,
            pb,
            threshold,
        )

        for metric in [
            "Sensitivity",
            "Specificity",
            "PPV",
            "NPV",
        ]:
            samples[metric].append(
                cm[metric]
            )

    return {
        metric: percentile_ci(values)
        for metric, values in samples.items()
    }


# 6. Learn the Fixed 90%-Specificity Policy on CAL


In [ ]:
threshold_rows = []

for landmark in LANDMARK_ORDER:
    for model_id in MODEL_ORDER[landmark]:
        df = predictions[landmark][model_id]

        cal = df.loc[
            df["split"].eq("cal")
        ].copy()

        result = choose_threshold_for_specificity(
            cal[Y_COL].to_numpy(),
            cal[PRIMARY_PROB_COL].to_numpy(),
            target_specificity=TARGET_SPECIFICITY,
        )

        threshold_rows.append(
            {
                "landmark": landmark,
                "model_id": model_id,
                "probability_scale": PRIMARY_PROB_COL,
                "target_specificity": TARGET_SPECIFICITY,
                **result,
                "cal_n": len(cal),
                "cal_cases": int(cal[Y_COL].sum()),
                "cal_controls": int(
                    len(cal) - cal[Y_COL].sum()
                ),
            }
        )

threshold_df = pd.DataFrame(
    threshold_rows
)

THRESHOLD_PATH = (
    f"{RESULT_DIR}/"
    f"fixed_policy_thresholds_cal_{FINAL_DATE}.csv"
)

save_gcs_csv(
    threshold_df,
    THRESHOLD_PATH,
)

print(
    threshold_df.to_string(
        index=False
    )
)


# 7. TEST Performance with 95% Confidence Intervals


In [ ]:
performance_rows = []

for landmark in LANDMARK_ORDER:
    for model_id in MODEL_ORDER[landmark]:
        df = predictions[landmark][model_id]

        test = df.loc[
            df["split"].eq("test")
        ].copy()

        y = test[Y_COL].to_numpy()
        p = test[PRIMARY_PROB_COL].to_numpy()

        threshold = float(
            threshold_df.loc[
                threshold_df["landmark"].eq(landmark)
                & threshold_df["model_id"].eq(model_id),
                "threshold",
            ].iloc[0]
        )

        cm = confusion_metrics(
            y,
            p,
            threshold,
        )

        intercept, slope = calibration_intercept_slope(
            y,
            p,
        )

        citl = calibration_in_the_large(
            y,
            p,
        )

        point = {
            "AUROC": safe_auroc(y, p),
            "AUPRC": safe_auprc(y, p),
            "Brier": brier_score_loss(y, p),
            "O_E": oe_ratio(y, p),
            "Calibration_intercept": intercept,
            "Calibration_slope": slope,
            "CITL": citl,
            **cm,
        }

        ci = bootstrap_test_metrics(
            y,
            p,
            threshold,
            n_boot=N_BOOT,
            seed=BOOT_SEED,
        )

        row = {
            "landmark": landmark,
            "model_id": model_id,
            "probability_scale": PRIMARY_PROB_COL,
            "n_test": len(test),
            "n_cases": int(y.sum()),
            "n_controls": int(len(y) - y.sum()),
            "threshold_cal": threshold,
        }

        for metric, value in point.items():
            row[metric] = value

            if metric in ci:
                row[f"{metric}_ci_low"] = ci[metric][0]
                row[f"{metric}_ci_high"] = ci[metric][1]

        performance_rows.append(row)

performance_df = pd.DataFrame(
    performance_rows
)

PERFORMANCE_PATH = (
    f"{RESULT_DIR}/"
    f"test_performance_all_models_{FINAL_DATE}.csv"
)

save_gcs_csv(
    performance_df,
    PERFORMANCE_PATH,
)

show_cols = [
    "landmark",
    "model_id",
    "n_test",
    "n_cases",
    "AUROC",
    "AUROC_ci_low",
    "AUROC_ci_high",
    "AUPRC",
    "AUPRC_ci_low",
    "AUPRC_ci_high",
    "Sensitivity",
    "Sensitivity_ci_low",
    "Sensitivity_ci_high",
    "Specificity",
    "Specificity_ci_low",
    "Specificity_ci_high",
    "PPV",
    "NPV",
    "Brier",
]

print(
    performance_df[
        show_cols
    ].to_string(
        index=False
    )
)


# 8. Raw-vs-Calibrated Discrimination Audit


In [ ]:
discrimination_scale_rows = []

for landmark in LANDMARK_ORDER:
    for model_id in MODEL_ORDER[landmark]:
        test = predictions[landmark][model_id].loc[
            predictions[landmark][model_id]["split"].eq("test")
        ]

        y = test[Y_COL].to_numpy()

        for prob_col in ["p_raw", "p_calibrated"]:
            p = test[prob_col].to_numpy()

            discrimination_scale_rows.append(
                {
                    "landmark": landmark,
                    "model_id": model_id,
                    "probability_scale": prob_col,
                    "AUROC": safe_auroc(y, p),
                    "AUPRC": safe_auprc(y, p),
                    "Brier": brier_score_loss(y, p),
                }
            )

discrimination_scale_df = pd.DataFrame(
    discrimination_scale_rows
)

save_gcs_csv(
    discrimination_scale_df,
    f"{RESULT_DIR}/"
    f"raw_vs_calibrated_discrimination_{FINAL_DATE}.csv",
)

print(
    discrimination_scale_df.to_string(
        index=False
    )
)


# 9. Paired AUROC Comparison: DeLong Test


In [ ]:
def compute_midrank(x):
    x = np.asarray(x)
    order = np.argsort(x)
    sorted_x = x[order]

    n = len(x)
    midranks = np.zeros(n, dtype=float)

    i = 0
    while i < n:
        j = i
        while j < n and sorted_x[j] == sorted_x[i]:
            j += 1

        midrank = 0.5 * (i + j - 1) + 1
        midranks[i:j] = midrank
        i = j

    out = np.empty(n, dtype=float)
    out[order] = midranks
    return out

def fast_delong(predictions_sorted_transposed, label_1_count):
    m = label_1_count
    n = predictions_sorted_transposed.shape[1] - m
    k = predictions_sorted_transposed.shape[0]

    positive_examples = predictions_sorted_transposed[:, :m]
    negative_examples = predictions_sorted_transposed[:, m:]

    tx = np.empty((k, m))
    ty = np.empty((k, n))
    tz = np.empty((k, m + n))

    for r in range(k):
        tx[r, :] = compute_midrank(
            positive_examples[r, :]
        )

        ty[r, :] = compute_midrank(
            negative_examples[r, :]
        )

        tz[r, :] = compute_midrank(
            predictions_sorted_transposed[r, :]
        )

    aucs = (
        tz[:, :m].sum(axis=1) / m / n
        - (m + 1.0) / (2.0 * n)
    )

    v01 = (
        tz[:, :m]
        - tx
    ) / n

    v10 = 1.0 - (
        tz[:, m:]
        - ty
    ) / m

    sx = np.cov(v01)
    sy = np.cov(v10)

    delong_cov = sx / m + sy / n

    return aucs, delong_cov

def delong_roc_test(y_true, p1, p2):
    y_true = np.asarray(y_true).astype(int)
    p1 = np.asarray(p1, dtype=float)
    p2 = np.asarray(p2, dtype=float)

    order = np.argsort(-y_true)
    label_1_count = int(y_true.sum())

    preds = np.vstack(
        [p1, p2]
    )[:, order]

    aucs, covariance = fast_delong(
        preds,
        label_1_count,
    )

    delta = float(
        aucs[1] - aucs[0]
    )

    var_delta = (
        covariance[0, 0]
        + covariance[1, 1]
        - 2 * covariance[0, 1]
    )

    se = math.sqrt(
        max(var_delta, 0.0)
    )

    if se == 0:
        z = np.inf if delta != 0 else 0.0
        p_value = 0.0 if delta != 0 else 1.0
    else:
        z = delta / se
        p_value = 2 * stats.norm.sf(
            abs(z)
        )

    ci_low = delta - 1.96 * se
    ci_high = delta + 1.96 * se

    return {
        "auc_earlier": float(aucs[0]),
        "auc_later": float(aucs[1]),
        "delta_auroc": delta,
        "delta_auroc_ci_low": float(ci_low),
        "delta_auroc_ci_high": float(ci_high),
        "delong_z": float(z),
        "delong_p": float(p_value),
    }


# 10. Paired PR-AUC Difference by Bootstrap


In [ ]:
def paired_bootstrap_metric_difference(
    y,
    p1,
    p2,
    metric_fn,
    *,
    n_boot=N_BOOT,
    seed=BOOT_SEED,
):
    y = np.asarray(y).astype(int)
    p1 = np.asarray(p1, dtype=float)
    p2 = np.asarray(p2, dtype=float)

    rng = np.random.default_rng(seed)

    point1 = metric_fn(y, p1)
    point2 = metric_fn(y, p2)
    point_delta = point2 - point1

    deltas = []

    for _ in range(n_boot):
        idx = stratified_bootstrap_indices(
            y,
            rng,
        )

        m1 = metric_fn(
            y[idx],
            p1[idx],
        )

        m2 = metric_fn(
            y[idx],
            p2[idx],
        )

        deltas.append(
            m2 - m1
        )

    ci_low, ci_high = percentile_ci(
        deltas
    )

    deltas = np.asarray(
        deltas,
        dtype=float,
    )

    # Two-sided paired bootstrap sign test approximation.
    p_lower = np.mean(
        deltas <= 0
    )
    p_upper = np.mean(
        deltas >= 0
    )

    p_value = min(
        1.0,
        2 * min(
            p_lower,
            p_upper,
        ),
    )

    return {
        "metric_earlier": float(point1),
        "metric_later": float(point2),
        "delta": float(point_delta),
        "ci_low": float(ci_low),
        "ci_high": float(ci_high),
        "p_value": float(p_value),
    }


# 11. McNemar Test and Case Reclassification


In [ ]:
def exact_mcnemar_correctness(
    y,
    pred1,
    pred2,
):
    y = np.asarray(y).astype(int)
    pred1 = np.asarray(pred1).astype(int)
    pred2 = np.asarray(pred2).astype(int)

    correct1 = pred1 == y
    correct2 = pred2 == y

    earlier_only_correct = int(
        np.sum(
            correct1 & ~correct2
        )
    )

    later_only_correct = int(
        np.sum(
            ~correct1 & correct2
        )
    )

    discordant = (
        earlier_only_correct
        + later_only_correct
    )

    if discordant == 0:
        p = 1.0
    else:
        p = stats.binomtest(
            min(
                earlier_only_correct,
                later_only_correct,
            ),
            n=discordant,
            p=0.5,
            alternative="two-sided",
        ).pvalue

    return {
        "discordant_pairs": discordant,
        "earlier_only_correct": earlier_only_correct,
        "later_only_correct": later_only_correct,
        "mcnemar_p": float(p),
    }

def case_reclassification(y, pred1, pred2):
    y = np.asarray(y).astype(int)
    pred1 = np.asarray(pred1).astype(int)
    pred2 = np.asarray(pred2).astype(int)

    case = y == 1

    newly_detected = int(
        np.sum(
            case
            & (pred1 == 0)
            & (pred2 == 1)
        )
    )

    lost_detection = int(
        np.sum(
            case
            & (pred1 == 1)
            & (pred2 == 0)
        )
    )

    total_cases = int(
        case.sum()
    )

    net_gain = (
        newly_detected
        - lost_detection
    )

    return {
        "n_cases": total_cases,
        "newly_detected_cases": newly_detected,
        "lost_detected_cases": lost_detection,
        "net_case_gain": net_gain,
        "net_case_gain_pct": (
            100 * net_gain / total_cases
            if total_cases else np.nan
        ),
    }


# 12. Consecutive-Layer Statistical Comparisons


In [ ]:
comparison_rows = []
reclass_rows = []

for landmark in LANDMARK_ORDER:
    order = MODEL_ORDER[landmark]

    for earlier_id, later_id in zip(
        order[:-1],
        order[1:],
    ):
        earlier = (
            predictions[landmark][earlier_id]
            .loc[
                predictions[landmark][earlier_id]["split"].eq("test")
            ]
            .sort_values(ID_COL)
            .reset_index(drop=True)
        )

        later = (
            predictions[landmark][later_id]
            .loc[
                predictions[landmark][later_id]["split"].eq("test")
            ]
            .sort_values(ID_COL)
            .reset_index(drop=True)
        )

        if not earlier[
            [ID_COL, Y_COL]
        ].equals(
            later[
                [ID_COL, Y_COL]
            ]
        ):
            raise ValueError(
                f"{landmark}: paired TEST rows differ "
                f"for {earlier_id}->{later_id}."
            )

        y = earlier[Y_COL].to_numpy()
        p1 = earlier[PRIMARY_PROB_COL].to_numpy()
        p2 = later[PRIMARY_PROB_COL].to_numpy()

        delong = delong_roc_test(
            y,
            p1,
            p2,
        )

        pr = paired_bootstrap_metric_difference(
            y,
            p1,
            p2,
            safe_auprc,
            n_boot=N_BOOT,
            seed=BOOT_SEED,
        )

        thr1 = float(
            threshold_df.loc[
                threshold_df["landmark"].eq(landmark)
                & threshold_df["model_id"].eq(earlier_id),
                "threshold",
            ].iloc[0]
        )

        thr2 = float(
            threshold_df.loc[
                threshold_df["landmark"].eq(landmark)
                & threshold_df["model_id"].eq(later_id),
                "threshold",
            ].iloc[0]
        )

        pred1 = (
            p1 >= thr1
        ).astype(int)

        pred2 = (
            p2 >= thr2
        ).astype(int)

        mc = exact_mcnemar_correctness(
            y,
            pred1,
            pred2,
        )

        reclass = case_reclassification(
            y,
            pred1,
            pred2,
        )

        comparison_rows.append(
            {
                "landmark": landmark,
                "transition": f"{earlier_id} -> {later_id}",
                "earlier_model": earlier_id,
                "later_model": later_id,
                **delong,
                "auprc_earlier": pr["metric_earlier"],
                "auprc_later": pr["metric_later"],
                "delta_auprc": pr["delta"],
                "delta_auprc_ci_low": pr["ci_low"],
                "delta_auprc_ci_high": pr["ci_high"],
                "auprc_p_bootstrap": pr["p_value"],
                **mc,
            }
        )

        reclass_rows.append(
            {
                "landmark": landmark,
                "transition": f"{earlier_id} -> {later_id}",
                "earlier_model": earlier_id,
                "later_model": later_id,
                "threshold_earlier": thr1,
                "threshold_later": thr2,
                **reclass,
            }
        )

comparison_df = pd.DataFrame(
    comparison_rows
)

reclassification_df = pd.DataFrame(
    reclass_rows
)

save_gcs_csv(
    comparison_df,
    f"{RESULT_DIR}/"
    f"paired_model_comparisons_{FINAL_DATE}.csv",
)

save_gcs_csv(
    reclassification_df,
    f"{RESULT_DIR}/"
    f"case_reclassification_{FINAL_DATE}.csv",
)

print("\nPAIRED MODEL COMPARISONS")
print(
    comparison_df.to_string(
        index=False
    )
)

print("\nCASE RECLASSIFICATION")
print(
    reclassification_df.to_string(
        index=False
    )
)


# 13. Manuscript-Formatted Performance Table


In [ ]:
def fmt_est_ci(est, low, high, digits=3):
    if pd.isna(est):
        return ""
    if pd.isna(low) or pd.isna(high):
        return f"{est:.{digits}f}"
    return (
        f"{est:.{digits}f} "
        f"({low:.{digits}f} to {high:.{digits}f})"
    )

manuscript_rows = []

for _, r in performance_df.iterrows():
    manuscript_rows.append(
        {
            "Landmark": r["landmark"],
            "Model": r["model_id"],
            "N": int(r["n_test"]),
            "Cases": int(r["n_cases"]),
            "AUROC (95% CI)": fmt_est_ci(
                r["AUROC"],
                r["AUROC_ci_low"],
                r["AUROC_ci_high"],
            ),
            "PR-AUC (95% CI)": fmt_est_ci(
                r["AUPRC"],
                r["AUPRC_ci_low"],
                r["AUPRC_ci_high"],
            ),
            "Sensitivity (95% CI)": fmt_est_ci(
                r["Sensitivity"],
                r["Sensitivity_ci_low"],
                r["Sensitivity_ci_high"],
            ),
            "Specificity (95% CI)": fmt_est_ci(
                r["Specificity"],
                r["Specificity_ci_low"],
                r["Specificity_ci_high"],
            ),
            "PPV (95% CI)": fmt_est_ci(
                r["PPV"],
                r["PPV_ci_low"],
                r["PPV_ci_high"],
            ),
            "NPV (95% CI)": fmt_est_ci(
                r["NPV"],
                r["NPV_ci_low"],
                r["NPV_ci_high"],
            ),
            "Brier score (95% CI)": fmt_est_ci(
                r["Brier"],
                r["Brier_ci_low"],
                r["Brier_ci_high"],
            ),
            "CAL threshold": f"{r['threshold_cal']:.6f}",
        }
    )

manuscript_performance_df = pd.DataFrame(
    manuscript_rows
)

save_gcs_csv(
    manuscript_performance_df,
    f"{RESULT_DIR}/"
    f"manuscript_performance_table_{FINAL_DATE}.csv",
)

display(
    manuscript_performance_df
)


# 14. Manuscript-Formatted Consecutive Model Comparison Table


In [ ]:
def format_p(p):
    if pd.isna(p):
        return ""
    if p < 1e-300:
        return "<1 × 10⁻³⁰⁰"
    if p < 0.001:
        return f"{p:.2e}"
    return f"{p:.4f}"

comparison_fmt_rows = []

for _, r in comparison_df.iterrows():
    comparison_fmt_rows.append(
        {
            "Landmark": r["landmark"],
            "Transition": r["transition"],
            "Discordant pairs": int(r["discordant_pairs"]),
            "AUROC (earlier)": f"{r['auc_earlier']:.3f}",
            "AUROC (later)": f"{r['auc_later']:.3f}",
            "ΔAUROC (95% CI)": (
                f"{r['delta_auroc']:+.3f} "
                f"({r['delta_auroc_ci_low']:+.3f} to "
                f"{r['delta_auroc_ci_high']:+.3f})"
            ),
            "DeLong p value": format_p(
                r["delong_p"]
            ),
            "PR-AUC (earlier)": f"{r['auprc_earlier']:.3f}",
            "PR-AUC (later)": f"{r['auprc_later']:.3f}",
            "ΔPR-AUC (95% CI)": (
                f"{r['delta_auprc']:+.3f} "
                f"({r['delta_auprc_ci_low']:+.3f} to "
                f"{r['delta_auprc_ci_high']:+.3f})"
            ),
            "PR-AUC p value": format_p(
                r["auprc_p_bootstrap"]
            ),
            "McNemar p value": format_p(
                r["mcnemar_p"]
            ),
        }
    )

comparison_fmt_df = pd.DataFrame(
    comparison_fmt_rows
)

save_gcs_csv(
    comparison_fmt_df,
    f"{RESULT_DIR}/"
    f"manuscript_paired_comparison_table_{FINAL_DATE}.csv",
)

display(
    comparison_fmt_df
)


# 15. Quantitative Calibration Summary


In [ ]:
calibration_summary_cols = [
    "landmark",
    "model_id",
    "n_test",
    "n_cases",
    "Brier",
    "Brier_ci_low",
    "Brier_ci_high",
    "CITL",
    "CITL_ci_low",
    "CITL_ci_high",
    "Calibration_intercept",
    "Calibration_intercept_ci_low",
    "Calibration_intercept_ci_high",
    "Calibration_slope",
    "Calibration_slope_ci_low",
    "Calibration_slope_ci_high",
]

calibration_summary_df = performance_df[
    calibration_summary_cols
].copy()

save_gcs_csv(
    calibration_summary_df,
    f"{RESULT_DIR}/"
    f"test_calibration_metrics_{FINAL_DATE}.csv",
)

display(calibration_summary_df)


# 16. Operating-Point Sensitivity Analysis: 80%, 85%, 90%, and 95% Specificity


In [ ]:
policy_rows = []

def evaluate_policy_set(
    df,
    *,
    landmark,
    model_id,
    analysis_group,
):
    cal = df.loc[
        df["split"].eq("cal")
    ].copy()

    test = df.loc[
        df["split"].eq("test")
    ].copy()

    for target_spec in SPEC_LEVELS:
        threshold_info = choose_threshold_for_specificity(
            cal[Y_COL].to_numpy(),
            cal[PRIMARY_PROB_COL].to_numpy(),
            target_specificity=target_spec,
        )

        threshold = threshold_info["threshold"]

        cm = confusion_metrics(
            test[Y_COL].to_numpy(),
            test[PRIMARY_PROB_COL].to_numpy(),
            threshold,
        )

        policy_rows.append(
            {
                "analysis_group": analysis_group,
                "landmark": landmark,
                "model_id": model_id,
                "target_specificity": target_spec,
                "threshold_cal": threshold,
                "cal_specificity": threshold_info[
                    "cal_specificity"
                ],
                "cal_sensitivity": threshold_info[
                    "cal_sensitivity"
                ],
                "n_test": len(test),
                "n_cases": int(test[Y_COL].sum()),
                **cm,
            }
        )

for landmark in LANDMARK_ORDER:
    for model_id in MODEL_ORDER[landmark]:
        evaluate_policy_set(
            predictions[landmark][model_id],
            landmark=landmark,
            model_id=model_id,
            analysis_group="primary_layered",
        )

for label, df in comparator_predictions.items():
    evaluate_policy_set(
        df,
        landmark="1y",
        model_id=label,
        analysis_group=(
            "clinical_benchmark"
            if label == "Clinical LR"
            else "algorithmic_sensitivity"
        ),
    )

policy_sensitivity_df = pd.DataFrame(
    policy_rows
)

save_gcs_csv(
    policy_sensitivity_df,
    f"{RESULT_DIR}/"
    f"fixed_specificity_sensitivity_80_85_90_95_{FINAL_DATE}.csv",
)

display(
    policy_sensitivity_df[
        [
            "analysis_group",
            "landmark",
            "model_id",
            "target_specificity",
            "threshold_cal",
            "Sensitivity",
            "Specificity",
            "PPV",
            "NPV",
            "TP",
            "FP",
            "TN",
            "FN",
        ]
    ]
)


# 17. Paired Confidence Intervals for Sensitivity Differences


In [ ]:
def paired_sensitivity_difference_ci(
    y,
    p_earlier,
    p_later,
    threshold_earlier,
    threshold_later,
    *,
    n_boot=N_BOOT,
    seed=BOOT_SEED,
):
    y = np.asarray(y).astype(int)
    p_earlier = np.asarray(p_earlier, dtype=float)
    p_later = np.asarray(p_later, dtype=float)

    case_idx = np.where(y == 1)[0]

    pred_earlier = (
        p_earlier >= threshold_earlier
    ).astype(int)

    pred_later = (
        p_later >= threshold_later
    ).astype(int)

    point = float(
        pred_later[case_idx].mean()
        - pred_earlier[case_idx].mean()
    )

    rng = np.random.default_rng(seed)
    deltas = []

    for _ in range(n_boot):
        b = rng.choice(
            case_idx,
            size=len(case_idx),
            replace=True,
        )

        deltas.append(
            float(
                pred_later[b].mean()
                - pred_earlier[b].mean()
            )
        )

    lo, hi = percentile_ci(
        deltas
    )

    return point, lo, hi

sensitivity_delta_rows = []

for landmark in LANDMARK_ORDER:
    order = MODEL_ORDER[landmark]

    for target_spec in SPEC_LEVELS:
        for earlier_id, later_id in zip(
            order[:-1],
            order[1:],
        ):
            earlier = (
                predictions[landmark][earlier_id]
                .loc[
                    predictions[landmark][earlier_id]["split"].eq("test")
                ]
                .sort_values(ID_COL)
                .reset_index(drop=True)
            )

            later = (
                predictions[landmark][later_id]
                .loc[
                    predictions[landmark][later_id]["split"].eq("test")
                ]
                .sort_values(ID_COL)
                .reset_index(drop=True)
            )

            if not earlier[
                [ID_COL, Y_COL]
            ].equals(
                later[
                    [ID_COL, Y_COL]
                ]
            ):
                raise ValueError(
                    f"{landmark}: paired rows differ."
                )

            t1 = float(
                policy_sensitivity_df.loc[
                    policy_sensitivity_df[
                        "analysis_group"
                    ].eq("primary_layered")
                    & policy_sensitivity_df[
                        "landmark"
                    ].eq(landmark)
                    & policy_sensitivity_df[
                        "model_id"
                    ].eq(earlier_id)
                    & policy_sensitivity_df[
                        "target_specificity"
                    ].eq(target_spec),
                    "threshold_cal",
                ].iloc[0]
            )

            t2 = float(
                policy_sensitivity_df.loc[
                    policy_sensitivity_df[
                        "analysis_group"
                    ].eq("primary_layered")
                    & policy_sensitivity_df[
                        "landmark"
                    ].eq(landmark)
                    & policy_sensitivity_df[
                        "model_id"
                    ].eq(later_id)
                    & policy_sensitivity_df[
                        "target_specificity"
                    ].eq(target_spec),
                    "threshold_cal",
                ].iloc[0]
            )

            delta, lo, hi = paired_sensitivity_difference_ci(
                earlier[Y_COL].to_numpy(),
                earlier[PRIMARY_PROB_COL].to_numpy(),
                later[PRIMARY_PROB_COL].to_numpy(),
                t1,
                t2,
            )

            sensitivity_delta_rows.append(
                {
                    "landmark": landmark,
                    "transition": (
                        f"{earlier_id} -> {later_id}"
                    ),
                    "target_specificity": target_spec,
                    "delta_sensitivity": delta,
                    "delta_sensitivity_ci_low": lo,
                    "delta_sensitivity_ci_high": hi,
                }
            )

sensitivity_delta_df = pd.DataFrame(
    sensitivity_delta_rows
)

save_gcs_csv(
    sensitivity_delta_df,
    f"{RESULT_DIR}/"
    f"paired_sensitivity_differences_{FINAL_DATE}.csv",
)

display(sensitivity_delta_df)


# 18. One-Year Parsimonious Clinical Benchmark


In [ ]:
def evaluate_frozen_prediction_df(
    df,
    *,
    label,
    target_specificity=TARGET_SPECIFICITY,
):
    cal = df.loc[
        df["split"].eq("cal")
    ].copy()

    test = df.loc[
        df["split"].eq("test")
    ].copy()

    threshold_info = choose_threshold_for_specificity(
        cal[Y_COL].to_numpy(),
        cal[PRIMARY_PROB_COL].to_numpy(),
        target_specificity=target_specificity,
    )

    y = test[Y_COL].to_numpy()
    p = test[PRIMARY_PROB_COL].to_numpy()

    intercept, slope = calibration_intercept_slope(
        y,
        p,
    )

    return {
        "model": label,
        "N": len(test),
        "cases": int(y.sum()),
        "AUROC": safe_auroc(y, p),
        "AUPRC": safe_auprc(y, p),
        "Brier": brier_score_loss(y, p),
        "CITL": calibration_in_the_large(y, p),
        "Calibration_intercept": intercept,
        "Calibration_slope": slope,
        "threshold_cal": threshold_info["threshold"],
        **confusion_metrics(
            y,
            p,
            threshold_info["threshold"],
        ),
    }

clinical_row = evaluate_frozen_prediction_df(
    comparator_predictions["Clinical LR"],
    label="Clinical LR",
)

m2_row = evaluate_frozen_prediction_df(
    predictions["1y"]["M2"],
    label="M2 XGBoost",
)

clinical_benchmark_df = pd.DataFrame(
    [
        clinical_row,
        m2_row,
    ]
)

# Paired DeLong comparison.
clin_test = (
    comparator_predictions["Clinical LR"]
    .loc[
        comparator_predictions["Clinical LR"]["split"].eq("test")
    ]
    .sort_values(ID_COL)
    .reset_index(drop=True)
)

m2_test = (
    predictions["1y"]["M2"]
    .loc[
        predictions["1y"]["M2"]["split"].eq("test")
    ]
    .sort_values(ID_COL)
    .reset_index(drop=True)
)

clinical_vs_m2_delong = delong_roc_test(
    clin_test[Y_COL].to_numpy(),
    clin_test[PRIMARY_PROB_COL].to_numpy(),
    m2_test[PRIMARY_PROB_COL].to_numpy(),
)

save_gcs_csv(
    clinical_benchmark_df,
    f"{RESULT_DIR}/"
    f"clinical_baseline_vs_m2_1y_{FINAL_DATE}.csv",
)

save_gcs_json(
    clinical_vs_m2_delong,
    f"{RESULT_DIR}/"
    f"clinical_baseline_vs_m2_delong_1y_{FINAL_DATE}.json",
)

display(clinical_benchmark_df)
print("Clinical LR vs M2 DeLong:", clinical_vs_m2_delong)


# 19. One-Year Algorithmic Sensitivity: XGBoost vs Logistic Regression vs Random Forest


In [ ]:
algorithm_prediction_sets = {
    "XGBoost": predictions["1y"]["M5"],
    "Logistic Regression": comparator_predictions["M5 LR"],
    "Random Forest": comparator_predictions["M5 RF"],
}

algorithm_rows = []

for label, df in algorithm_prediction_sets.items():
    # Point estimates via the existing helper.
    point_row = evaluate_frozen_prediction_df(df, label=label)

    # Extract TEST y/p and CAL threshold for bootstrap.
    test = df.loc[df["split"].eq("test")].copy()
    y = test[Y_COL].to_numpy()
    p = test[PRIMARY_PROB_COL].to_numpy()
    threshold = point_row["threshold_cal"]

    # Reuse the same bootstrap helper used for M1–M5 in cell 18.
    ci = bootstrap_test_metrics(
        y, p, threshold,
        n_boot=N_BOOT,
        seed=BOOT_SEED,
    )

    for metric, (lo, hi) in ci.items():
        point_row[f"{metric}_ci_low"] = lo
        point_row[f"{metric}_ci_high"] = hi

    algorithm_rows.append(point_row)

algorithmic_sensitivity_df = pd.DataFrame(algorithm_rows)

save_gcs_csv(
    algorithmic_sensitivity_df,
    f"{RESULT_DIR}/"
    f"algorithmic_sensitivity_1y_M5_{FINAL_DATE}.csv",
)

# Pairwise DeLong vs XGBoost (unchanged).
xgb_test = (
    algorithm_prediction_sets["XGBoost"]
    .loc[
        algorithm_prediction_sets["XGBoost"]["split"].eq("test")
    ]
    .sort_values(ID_COL)
    .reset_index(drop=True)
)

delong_rows = []

for label in ["Logistic Regression", "Random Forest"]:
    other = (
        algorithm_prediction_sets[label]
        .loc[
            algorithm_prediction_sets[label]["split"].eq("test")
        ]
        .sort_values(ID_COL)
        .reset_index(drop=True)
    )

    d = delong_roc_test(
        xgb_test[Y_COL].to_numpy(),
        xgb_test[PRIMARY_PROB_COL].to_numpy(),
        other[PRIMARY_PROB_COL].to_numpy(),
    )

    delong_rows.append({"comparison": f"XGBoost vs {label}", **d})

algorithmic_delong_df = pd.DataFrame(delong_rows)

save_gcs_csv(
    algorithmic_delong_df,
    f"{RESULT_DIR}/"
    f"algorithmic_sensitivity_delong_1y_M5_{FINAL_DATE}.csv",
)

display(algorithmic_sensitivity_df)
display(algorithmic_delong_df)

In [ ]:
print(algorithmic_sensitivity_df.to_string())

In [ ]:
pwd

# 20. ROC and Precision-Recall Curves


In [ ]:
for landmark in LANDMARK_ORDER:
    # ROC
    plt.figure(figsize=(7, 6))

    for model_id in MODEL_ORDER[landmark]:
        test = (
            predictions[landmark][model_id]
            .loc[
                predictions[landmark][model_id]["split"].eq("test")
            ]
        )

        y = test[Y_COL].to_numpy()
        p = test[PRIMARY_PROB_COL].to_numpy()

        fpr, tpr, _ = roc_curve(
            y,
            p,
        )

        auc = safe_auroc(
            y,
            p,
        )

        plt.plot(
            fpr,
            tpr,
            label=f"{model_id} (AUROC={auc:.3f})",
        )

    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        linewidth=1,
    )

    plt.xlabel("1 - Specificity")
    plt.ylabel("Sensitivity")
    plt.title(f"ROC curves — {landmark}")
    plt.legend(
        loc="lower right",
        frameon=False,
    )
    plt.tight_layout()
    plt.show()

    # PR
    plt.figure(figsize=(7, 6))

    prevalence = None

    for model_id in MODEL_ORDER[landmark]:
        test = (
            predictions[landmark][model_id]
            .loc[
                predictions[landmark][model_id]["split"].eq("test")
            ]
        )

        y = test[Y_COL].to_numpy()
        p = test[PRIMARY_PROB_COL].to_numpy()

        precision, recall, _ = precision_recall_curve(
            y,
            p,
        )

        auprc = safe_auprc(
            y,
            p,
        )

        prevalence = y.mean()

        plt.plot(
            recall,
            precision,
            label=f"{model_id} (PR-AUC={auprc:.3f})",
        )

    plt.axhline(
        prevalence,
        linestyle="--",
        linewidth=1,
    )

    plt.xlabel("Recall (Sensitivity)")
    plt.ylabel("Precision (PPV)")
    plt.title(f"Precision-recall curves — {landmark}")
    plt.legend(
        loc="lower right",
        frameon=False,
    )
    plt.tight_layout()
    plt.show()


# 21. Calibration Curves


In [ ]:
def calibration_bins(y, p, n_bins=10):
    x = pd.DataFrame(
        {
            "y": np.asarray(y).astype(int),
            "p": np.asarray(p, dtype=float),
        }
    )

    # Rank-based qcut avoids failures from repeated isotonic probabilities.
    x["bin"] = pd.qcut(
        x["p"].rank(method="first"),
        q=n_bins,
        labels=False,
        duplicates="drop",
    )

    out = (
        x.groupby(
            "bin",
            observed=True,
        )
        .agg(
            mean_pred=("p", "mean"),
            observed=("y", "mean"),
            n=("y", "size"),
        )
        .reset_index()
    )

    return out

calibration_plot_rows = []

for landmark in LANDMARK_ORDER:
    plt.figure(figsize=(7, 6))

    for model_id in MODEL_ORDER[landmark]:
        test = (
            predictions[landmark][model_id]
            .loc[
                predictions[landmark][model_id]["split"].eq("test")
            ]
        )

        bins = calibration_bins(
            test[Y_COL].to_numpy(),
            test[PRIMARY_PROB_COL].to_numpy(),
            n_bins=10,
        )

        bins["landmark"] = landmark
        bins["model_id"] = model_id

        calibration_plot_rows.append(
            bins
        )

        plt.plot(
            bins["mean_pred"],
            bins["observed"],
            marker="o",
            label=model_id,
        )

    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        linewidth=1,
    )

    plt.xlabel("Mean predicted risk")
    plt.ylabel("Observed event proportion")
    plt.title(f"Calibration — {landmark}")
    plt.legend(
        loc="best",
        frameon=False,
    )
    plt.tight_layout()
    plt.show()

calibration_bins_df = pd.concat(
    calibration_plot_rows,
    ignore_index=True,
)

save_gcs_csv(
    calibration_bins_df,
    f"{RESULT_DIR}/"
    f"calibration_bins_{FINAL_DATE}.csv",
)


# 22. Final Evaluation Handoff


In [ ]:
summary = {
    "date": FINAL_DATE,
    "primary_probability": PRIMARY_PROB_COL,
    "threshold_policy": (
        "CAL-derived threshold with specificity >=0.90; "
        "threshold transported unchanged to TEST"
    ),
    "test_bootstrap_replicates": N_BOOT,
    "bootstrap_seed": BOOT_SEED,
    "paired_auroc_test": "DeLong",
    "paired_auprc_test": (
        "paired stratified bootstrap difference"
    ),
    "classification_comparison": (
        "exact McNemar test on paired correctness"
    ),
    "case_reclassification": (
        "newly detected vs lost detected cases at model-specific "
        "CAL-derived fixed-policy thresholds"
    ),
}

save_gcs_json(
    summary,
    f"{RESULT_DIR}/"
    f"performance_analysis_summary_{FINAL_DATE}.json",
)

print("=" * 90)
print("NOTEBOOK 04 PERFORMANCE EVALUATION COMPLETE")
print("=" * 90)

print("\nKey outputs:")
for path in [
    f"{RESULT_DIR}/test_calibration_metrics_{FINAL_DATE}.csv",
    f"{RESULT_DIR}/fixed_specificity_sensitivity_80_85_90_95_{FINAL_DATE}.csv",
    f"{RESULT_DIR}/paired_sensitivity_differences_{FINAL_DATE}.csv",
    f"{RESULT_DIR}/clinical_baseline_vs_m2_1y_{FINAL_DATE}.csv",
    f"{RESULT_DIR}/algorithmic_sensitivity_1y_M5_{FINAL_DATE}.csv",
    THRESHOLD_PATH,
    PERFORMANCE_PATH,
    f"{RESULT_DIR}/paired_model_comparisons_{FINAL_DATE}.csv",
    f"{RESULT_DIR}/case_reclassification_{FINAL_DATE}.csv",
    f"{RESULT_DIR}/manuscript_performance_table_{FINAL_DATE}.csv",
    f"{RESULT_DIR}/manuscript_paired_comparison_table_{FINAL_DATE}.csv",
    f"{RESULT_DIR}/calibration_bins_{FINAL_DATE}.csv",
]:
    print(" ", path)
